# 10 — Evidence-Grounded Prompting and RAG Interfaces

## Scenario
Northstar must answer questions about refund policies. We want to avoid hallucination, so we require the model to ground its answers in factual evidence.

**The Problem:** LLMs are eager to please and will often invent plausible-sounding policies if they don't know the answer.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab10 import build_requests, is_abstention, retrieve
from northstar.evidence import check_citations


def show_request(request):
    print("SYSTEM:\n", request.system)
    for message in request.messages:
        if message.text:
            print(f"{message.role.upper()}:\n{message.text}")
        for part in message.parts:
            print(f"{message.role.upper()} PART:", part)

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: Ungrounded Generation (Baseline)

Watch what happens when we ask a niche question without any grounding.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i10/ungrounded/custom-mug")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
print("PARSED ANSWER:", response.text)


## Step 2: Manual Grounding (Classic RAG)

We "retrieve" a document (mocked here) and strictly instruct the model to use it.


In [ ]:
evidence = retrieve(client, "custom mug Northstar logo")
print("RETRIEVED:", evidence)
request = next(r for r in build_requests() if r.case_id == "i10/final/custom-mug")
show_request(request)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
report = check_citations(response.text, evidence)
print("PARSED CITATION REPORT:", report)
assert report.unknown_ids == set()
assert "[POL-992]" in response.text


## Step 3: Managed Grounding with a Retrieval Tool (State of the Art)

Instead of manually pasting text into prompts, an application can expose a retrieval tool and keep evidence selection explicit. In replay mode this tool call and its response are deterministic; live mode can connect it to an approved provider or search service.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i10/tool/custom-mug")
show_request(request)
tool_response = client.generate(request)
print("RECORDED TOOL CALL:", tool_response.tool_calls)
print("PARSED TOOL CALL:", tool_response.tool_calls[0].name)
assert tool_response.tool_calls[0].name == "search_policies"
answer_request = next(r for r in build_requests() if r.case_id == "i10/final/no-support")
show_request(answer_request)
answer = client.generate(answer_request)
print("RECORDED ABSTENTION:", answer.text)
print("PARSED ABSTENTION:", is_abstention(answer.text))
assert is_abstention(answer.text)


## Takeaway
The recorded grounding run cited `[POL-992]`, detected no unsupported citation in the supported answer, and abstained 1/1 when the policy evidence did not support an answer.


## References
- [Core Concepts & Workflow](README.md#core-concepts--workflow)
- [Deep dive](README.md#deep-dive)
- [Lab walkthrough](README.md#lab-walkthrough)
